# Inspect One Event Chunk Batch - v22

Loads one v22 quote/trade event-chunk batch, checks tensor shapes and finite values, optionally loads a checkpoint, runs one inference, and computes the binary-magnitude BCE loss.

In [ ]:
from pathlib import Path
import os

MODEL_VERSION = "v22"


def find_workspace(model_version: str) -> Path:
    """Find the repo/runtime root on either the laptop checkout or workstation runtime."""
    candidates: list[Path] = []
    for env_name in ("QUANT_RESEARCH_WORKSPACE", "QRW_WORKSPACE"):
        value = os.environ.get(env_name)
        if value:
            candidates.append(Path(value))

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend(
        [
            Path(r"D:/TradingCodes/quant-research-workbench-v22-runtime"),
            Path(r"D:/TradingCodes/quant-research-workbench"),
            Path(r"//DESKTOP-SAAI85T/Workstation-D/TradingCodes/quant-research-workbench-v22-runtime"),
        ]
    )

    checked: list[str] = []
    for candidate in candidates:
        root = candidate.resolve() if candidate.exists() else candidate
        marker = root / "research" / "inhouse_transformer" / model_version / "config.py"
        checked.append(str(marker))
        if marker.exists():
            return root
    raise FileNotFoundError(
        "Could not find a repo root containing "
        f"research/inhouse_transformer/{model_version}/config.py. "
        "Set QUANT_RESEARCH_WORKSPACE to the checkout/runtime root. Checked: "
        + "; ".join(checked[:12])
    )


WORKSPACE = find_workspace(MODEL_VERSION)
NOTEBOOK_DIR = WORKSPACE / "research" / "inhouse_transformer" / MODEL_VERSION
MODEL_ROOT = Path(r"D:/TradingData/quant-research-workbench/market_data/models/inhouse_transformer") / MODEL_VERSION
print(f"WORKSPACE={WORKSPACE}")
print(f"NOTEBOOK_DIR={NOTEBOOK_DIR}")

# Edit these values for the batch/checkpoint you want to inspect.
CHECKPOINT_PATH = ""  # e.g. r"D:/TradingData/.../models/inhouse_transformer/{version}/run/last.pt"
SESSION_DATE = "2025-11-03"
TICKERS = ("A",)
BATCH_SIZE = 4
DEVICE = "cuda"  # use "cpu" if CUDA is not available

FLATFILES_ROOT = Path(r"D:/market-data/flatfiles/us_stocks_sip")
CANONICAL_ROOT = Path(r"D:/market-data/flatfiles/us_stocks_sip/derived/canonical_events_v1")
CACHE_ROOT = Path(r"D:/market-data/flatfiles/us_stocks_sip/derived/event_chunks_v1")


In [ ]:

import sys
from dataclasses import fields
from pathlib import Path

import numpy as np
import pandas as pd
import torch

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.inhouse_transformer.v22.config import DataConfig, ModelConfig
from research.inhouse_transformer.v22.data import (
    CHUNK_SUMMARY_COLUMNS,
    QUOTE_FEATURE_COLUMNS,
    TRADE_FEATURE_COLUMNS,
    EventChunkDataset,
    available_sessions,
)
from research.inhouse_transformer.v22.model import HierarchicalEventTransformer, forecast_loss
from research.inhouse_transformer.v22.targets import target_values_to_bps


def dataclass_from_dict(cls, payload):
    allowed = {field.name for field in fields(cls)}
    values = {key: value for key, value in dict(payload or {}).items() if key in allowed}
    for key in ("flatfiles_root", "canonical_root", "cache_root"):
        if key in values:
            values[key] = Path(values[key])
    for key in ("tickers", "target_columns", "target_cache_horizon_chunks"):
        if key in values and isinstance(values[key], list):
            values[key] = tuple(values[key])
    return cls(**values)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu") if CHECKPOINT_PATH else None
if checkpoint and "config" in checkpoint:
    data_config = dataclass_from_dict(DataConfig, checkpoint["config"].get("data", {}))
    model_config = dataclass_from_dict(ModelConfig, checkpoint["config"].get("model", {}))
else:
    data_config = DataConfig()
    model_config = ModelConfig()

data_config.flatfiles_root = FLATFILES_ROOT
data_config.canonical_root = CANONICAL_ROOT
data_config.cache_root = CACHE_ROOT
data_config.tickers = TICKERS
model_config.target_bit_count = 1 + int(data_config.binary_magnitude_bits)

print(data_config)
print(model_config)
print("target horizons seconds:", data_config.target_horizon_seconds)


In [ ]:

sessions = available_sessions(data_config.flatfiles_root, SESSION_DATE, SESSION_DATE)
dataset = EventChunkDataset(
    config=data_config,
    sessions=sessions,
    tickers=TICKERS,
    batch_size=BATCH_SIZE,
    seed=17,
    mode="inspect",
    epochs=1,
    max_windows=BATCH_SIZE,
    shuffle=False,
)

batch = next(iter(dataset))
print("Batch keys and shapes:")
for key, value in batch.items():
    if torch.is_tensor(value):
        finite = torch.isfinite(value.float()).all().item() if value.numel() else True
        print(f"  {key:24s} shape={tuple(value.shape)} dtype={value.dtype} finite={finite}")
    else:
        print(f"  {key:24s} len={len(value)} sample={value[:3]}")


In [ ]:

model = HierarchicalEventTransformer(
    quote_feature_count=len(QUOTE_FEATURE_COLUMNS),
    trade_feature_count=len(TRADE_FEATURE_COLUMNS),
    chunk_summary_count=len(CHUNK_SUMMARY_COLUMNS),
    context_chunks=data_config.context_chunks,
    max_quote_events=data_config.max_quote_events,
    max_trade_events=data_config.max_trade_events,
    max_total_events=data_config.max_total_events,
    horizon_steps=data_config.target_horizon_count,
    target_count=len(data_config.target_columns),
    config=model_config,
)
if checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    print(f"Loaded checkpoint step={checkpoint.get('global_step', 'unknown')} from {CHECKPOINT_PATH}")

device = torch.device(DEVICE if DEVICE == "cuda" and torch.cuda.is_available() else "cpu")
model = model.to(device).eval()
model_inputs = (
    batch["quote_values"][:1].to(device),
    batch["trade_values"][:1].to(device),
    batch["event_kinds"][:1].to(device),
    batch["event_indices"][:1].to(device),
    batch["chunk_summary"][:1].to(device),
)
with torch.no_grad():
    prediction = model(
        batch["quote_values"].to(device),
        batch["trade_values"].to(device),
        batch["event_kinds"].to(device),
        batch["event_indices"].to(device),
        batch["chunk_summary"].to(device),
    )
    loss, loss_parts = forecast_loss(prediction, batch["targets"].to(device))
print("prediction", tuple(prediction.shape), "target", tuple(batch["targets"].shape))
print("loss", float(loss.detach().cpu()), loss_parts)


In [ ]:

# Keras-style model summary and graph view.
# Optional packages for richer output:
#   pip install torchinfo torchview graphviz
# Graph rendering also needs the Graphviz dot executable on PATH.

from IPython.display import Image, Markdown, display
import shutil

param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parameters={param_count:,} trainable={trainable_count:,}")

try:
    from torchinfo import summary
    print(summary(model, input_data=model_inputs, depth=8, col_names=("input_size", "output_size", "num_params"), verbose=0))
except Exception as exc:
    print("torchinfo summary unavailable:", repr(exc))
    print(model)

try:
    from torchview import draw_graph
    dot = shutil.which("dot")
    if dot is None:
        raise RuntimeError("Graphviz dot executable is not on PATH. Install Graphviz and add its bin folder to PATH.")
    graph = draw_graph(model, input_data=model_inputs, expand_nested=True, save_graph=True, filename=f"{MODEL_VERSION}_architecture", directory=str(NOTEBOOK_DIR))
    png_path = NOTEBOOK_DIR / f"{MODEL_VERSION}_architecture.png"
    display(Image(filename=str(png_path)))
except Exception as exc:
    display(Markdown(f"Graph rendering skipped: `{exc!r}`"))


In [ ]:

pred_np = prediction.detach().cpu().numpy()
target_np = batch["targets"].numpy()
current_mid = batch["current_mid"].numpy()
zeros = np.zeros_like(current_mid)
ones = np.ones_like(current_mid)
pred_bps = target_values_to_bps(pred_np, current_mid, zeros, ones, data_config.target_mode)
target_bps = batch["target_bps"].numpy()[..., 0]

rows = []
for horizon_index, horizon_seconds in enumerate(data_config.target_horizon_seconds):
    rows.append({
        "horizon_seconds": float(horizon_seconds),
        "target_bps": float(target_bps[0, horizon_index]),
        "pred_bps_hard_decode": float(pred_bps[0, horizon_index, 0]),
        "current_mid": float(current_mid[0]),
        "target_mid": float(batch["target_mid"][0, horizon_index, 0]),
        "pred_mid_hard_decode": float(current_mid[0] * np.exp(pred_bps[0, horizon_index, 0] / 10000.0)),
    })
pd.DataFrame(rows)
